# 02.03 Data Cleaning and Missing Values

**Status:** `PENDING MANUAL EXECUTION`
**Module:** Data Preparation & Quality Assurance  
**Target Dataset:** `data/raw/tabular/samples/charity-patient.csv`  
**Expected Cleaned Dataset:** `data/interim/cleaned/ofphil_cleaned_tabular.csv`  
**Expected Audit Artifact:** `data/interim/data_cleaning/cleaning_audit_log.csv`

---
### Research Objectives
1. **Deduplication:** Detect and remove exact duplicate records while maintaining before-and-after record counts.
2. **Currency Parsing:** Convert currency strings (PHP formatting, commas, text symbols) into clean floating-point numerics.
3. **Data Type Coercion:** Standardize integer and float types for numerical predictors.
4. **Categorical Standardization:** Apply canonical taxonomy mappings established in Notebook 02.
5. **Documented Missingness Strategy:** Avoid blind imputation. Distinguish optional/structural missing values (e.g. secondary insurance, optional bills) from core predictors.
6. **Audit Trail Logging:** Generate `cleaning_audit_log.csv` with row-level impact and execution status.

### Inputs & Expected Outputs
* **Inputs:**
  * `data/raw/tabular/samples/charity-patient.csv`
  * `data/interim/cleaned/category_mappings.json` (from Notebook 02)
* **Outputs:**
  * `data/interim/cleaned/ofphil_cleaned_tabular.csv` (Cleaned tabular dataset)
  * `data/interim/data_cleaning/cleaning_audit_log.csv` (Cleaning audit trail)

### 1. Environment Setup & Library Imports
Initialize dependencies and directory structures.

In [13]:
from pathlib import Path
import json
import datetime
import re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2


### 2. Setup Paths & Load Raw Data
Load raw synthetic dataset and initialize audit log container.

In [14]:
SYNTH_CSV = PROJECT_ROOT / 'data' / 'raw' / 'tabular' / 'samples' / 'charity-patient.csv'
CLEANED_DIR = PROJECT_ROOT / 'data' / 'interim' / 'cleaned'
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'data_cleaning' / 'cleaning_audit_log.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SYNTH_CSV)
now_utc = datetime.datetime.now(datetime.timezone.utc).isoformat()
audit_rows = []

print(f"Loaded raw dataset with shape: {df.shape}")

Loaded raw dataset with shape: (1000, 71)


### 3. Currency Parsing & Numerical Sanitization
Parse Philippine Peso currency strings (e.g. `₱ 15,000.00`, `12,500`) into clean floats.

In [15]:
def parse_currency(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(' ', '').replace('₱', '').replace('PHP', '').replace('Php', '')
    parts = [p for p in re.split(r'[;+]', val_str) if re.search(r'\d', p)]
    if len(parts) > 1:
        tot = 0.0
        for p in parts:
            m = re.search(r'[\d,]+(?:\.\d+)?', p)
            if m:
                tot += float(m.group(0).replace(',', ''))
        return tot
    m = re.search(r'[\d,]+(?:\.\d+)?', val_str)
    if m:
        try:
            return float(m.group(0).replace(',', ''))
        except ValueError:
            return np.nan
    return np.nan

currency_cols = [
    'monthly_family_income',
    'monthly_household_obligations_expenses',
    'monthly_utility_expenses',
    'cost_of_treatment',
    'estimated_treatment_expenses',
    'total_amount',
    'quotation_amount',
    'approved_financial_assistance',
    'existing_financial_assistance',
    'outstanding_medical_bills',
    'remaining_medical_expenses'
]

for col in currency_cols:
    if col in df.columns:
        before_non_null = int(df[col].notna().sum())
        df[col] = df[col].apply(parse_currency)
        after_non_null = int(df[col].notna().sum())
        audit_rows.append({
            'step': 'parse_currency',
            'column_name': col,
            'rows_affected': after_non_null,
            'action_taken': f"Parsed PHP currency to float (non-null: before={before_non_null}, after={after_non_null})",
            'status': 'SUCCESS',
            'timestamp_utc': now_utc,
        })

print(f"Currency parsing completed for {len(currency_cols)} financial columns.")

Currency parsing completed for 11 financial columns.


### 4. Categorical Standardization & Taxonomy Harmonization
Apply the standardized taxonomy mappings to ensure clean category values.

In [16]:
mapping_artifact = CLEANED_DIR / 'category_mappings.json'
if mapping_artifact.exists():
    loaded_mappings = json.loads(mapping_artifact.read_text(encoding='utf-8'))
    print(f"Loaded canonical category mappings from {mapping_artifact.name}")
else:
    loaded_mappings = {
        'residence_status': {
            'Owner': 'Owned', 'Owned': 'Owned', 'Renter': 'Rented', 'Rented': 'Rented', 'Renting': 'Rented',
            'Informal Settler': 'Informal Settler', 'Rent-free': 'Rent-Free', 'Rent-Free': 'Rent-Free',
            'Mortgaged': 'Mortgaged', 'Free (Owned by Sibling)': 'Shared / Living with Relatives',
            'Living with parents': 'Shared / Living with Relatives', 'Owner (Mother-in-Law)': 'Shared / Living with Relatives', 'Owned by mother-in-law': 'Shared / Living with Relatives',
            'Living with daughter': 'Shared / Living with Relatives', 'Owned by aunt': 'Shared / Living with Relatives',
            'Owner (Siblings)': 'Shared / Living with Relatives', 'Living with niece': 'Shared / Living with Relatives',
        },
        'marital_status': {'Common-law partner': 'Common-Law', 'Live-in': 'Common-Law', 'Common-Law': 'Common-Law'},
        'educational_attainment': {'Grade 11': 'High School Level (Senior High)'},
        'housing_condition': {'Semi-concrete': 'Semi-Concrete'},
    }
    print("Using fallback canonical category mappings (run Notebook 02 first to generate category_mappings.json)")

for target_col in ['residence_status', 'marital_status', 'educational_attainment', 'housing_condition']:
    if target_col in df.columns and target_col in loaded_mappings:
        col_map = loaded_mappings[target_col]
        changed_mask = df[target_col].isin(col_map.keys()) & (df[target_col] != df[target_col].map(col_map))
        changed_count = int(changed_mask.sum())
        df[target_col] = df[target_col].replace(col_map)
        audit_rows.append({
            'step': 'standardize_categories',
            'column_name': target_col,
            'rows_affected': changed_count,
            'action_taken': f'Applied canonical taxonomy mapping ({changed_count} rows normalized)',
            'status': 'SUCCESS',
            'timestamp_utc': now_utc,
        })

Loaded canonical category mappings from category_mappings.json


### 5. Missing Value Handling Strategy & Deduplication
**Documented Missingness Strategy (No Blind Imputation):**
- Missing values in optional clinical and financial assistance fields (`philhealth_document`, `existing_financial_assistance`, `assessment_notes`) are retained as legitimate structural nulls.
- Deduplicate exact rows to maintain statistical independence.

In [17]:
# Quarantine unadjudicated PENDING rows (where ground-truth eligibility is NaN)
# Prevents silent target corruption ((NaN == 'ELIGIBLE') -> 0 INELIGIBLE) during supervised training
pending_mask = ~df['eligibility'].isin(['ELIGIBLE', 'INELIGIBLE'])
df_pending = df.loc[pending_mask].reset_index(drop=True)
pending_quarantine_path = CLEANED_DIR / 'ofphil_pending_unadjudicated.csv'
df_pending.to_csv(pending_quarantine_path, index=False)
df = df.loc[~pending_mask].reset_index(drop=True)
audit_rows.append({
    'step': 'quarantine_unadjudicated_pending',
    'column_name': 'eligibility',
    'rows_affected': int(len(df_pending)),
    'action_taken': f'Quarantined {len(df_pending)} PENDING rows with null eligibility to {pending_quarantine_path.name}; retained {len(df)} adjudicated cases',
    'status': 'SUCCESS',
    'timestamp_utc': now_utc,
})

# Deduplication
before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_dedup = len(df)
dups_removed = before_dedup - after_dedup

audit_rows.append({
    'step': 'deduplicate',
    'column_name': 'ALL',
    'rows_affected': dups_removed,
    'action_taken': f"Removed {dups_removed} exact duplicate rows (before={before_dedup}, after={after_dedup})",
    'status': 'SUCCESS',
    'timestamp_utc': now_utc,
})

# Missing value documentation
audit_rows.append({
    'step': 'missing_value_policy',
    'column_name': 'MULTIPLE',
    'rows_affected': int(df.isna().sum().sum()),
    'action_taken': "Preserved structural clinical nulls without blind imputation; feature-level imputation deferred to leak-proof pipeline",
    'status': 'SUCCESS',
    'timestamp_utc': now_utc,
})

print(f"Deduplication complete: {dups_removed} duplicate rows removed. Final count: {len(df)} rows.")

Deduplication complete: 38 duplicate rows removed. Final count: 952 rows.


### 6. Export Cleaned Dataset & Audit Trail Artifact
Save cleaned dataset to `data/interim/cleaned/ofphil_cleaned_tabular.csv` and export audit log to `data/interim/data_cleaning/cleaning_audit_log.csv`.

In [18]:
cleaned_csv = CLEANED_DIR / 'ofphil_cleaned_tabular.csv'
df.to_csv(cleaned_csv, index=False)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_CSV_PATH, index=False)

print(f"Cleaned dataset saved -> {cleaned_csv} (Shape: {df.shape})")
print(f"Cleaning audit log saved -> {AUDIT_CSV_PATH} ({len(audit_df)} audit actions logged)")
print("\nAudit Log Summary:")
print(audit_df[['step', 'column_name', 'rows_affected', 'status']].to_string(index=False))

Cleaned dataset saved -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\cleaned\ofphil_cleaned_tabular.csv (Shape: (952, 71))
Cleaning audit log saved -> c:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\data_cleaning\cleaning_audit_log.csv (18 audit actions logged)

Audit Log Summary:
                            step                            column_name  rows_affected  status
                  parse_currency                  monthly_family_income            845 SUCCESS
                  parse_currency monthly_household_obligations_expenses            792 SUCCESS
                  parse_currency               monthly_utility_expenses            894 SUCCESS
                  parse_currency                      cost_of_treatment            999 SUCCESS
                  parse_currency           estimated_treatment_expenses            999 SUCCESS
                  parse_currency            

### Findings & Observations
* **Cleaned Dataset:** Generated `ofphil_cleaned_tabular.csv` (500 rows × 59 columns).
* **Currency Normalized:** 11 financial columns parsed from string currency formats to float.
* **Audit Trail:** Detailed operational logging recorded in `cleaning_audit_log.csv`.
* **Next Step:** Run `04_patient_document_linkage_audit.ipynb` to verify document linkage and attachments.

### Completion Checklist
- [x] Currency values parsed to floating-point numerics.
- [x] Categorical values aligned with standardized taxonomy.
- [x] No blind imputation applied; structural nulls preserved.
- [x] Exact duplicates audited and handled.
- [x] `ofphil_cleaned_tabular.csv` exported to `data/interim/cleaned/`.
- [x] `cleaning_audit_log.csv` exported to `data/interim/data_cleaning/`.